# 🔬 Eksperimen 2: Cross-Project Flaky Test Prediction pada Dataset Terimputasi (24 Pipelines)
**Kelompok Peneliti**: Kelompok A01 — RPL A (Magister Teknik Informatika ITS)  
**Dataset Acuan**: `Experiment_Ali/data/cleaned_test_features.csv` (Group-Aware Median Imputation, 0 Missing Values, 24 Proyek Utuh)  
**Hardware Target**: NVIDIA GeForce RTX 4050 Laptop GPU (CUDA 13.4, Driver 617.42)  
**Skenario**: Leave-One-Project-Out Cross-Validation (LOPO-CV, 24 Folds, Zero Data Leakage).

## 1. Verifikasi Lingkungan & GPU RTX 4050

In [ ]:
import sys
import os
from pathlib import Path

# Tambahkan direktori src ke sys.path
src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.append(src_dir)

import check_gpu
check_gpu.check_nvidia_smi()
check_gpu.check_python_environment()
check_gpu.check_xgboost_gpu()

## 2. Ingestion & Verifikasi Dataset Terimputasi (Cleaned)

In [ ]:
from data_loader import load_cleaned_dataset

df, features, projects = load_cleaned_dataset()
print(f"\n[STATUS] Total Baris: {len(df):,} | Proyek: {len(projects)} | Total Fitur: {len(features)}")
print(f"[STATUS] Flaky: {df['flaky'].sum():,} ({df['flaky'].mean()*100:.2f}%) | Non-Flaky: {(df['flaky']==0).sum():,} ({(df['flaky']==0).mean()*100:.2f}%)")
display(df[features].describe().T[['count', 'mean', 'std', 'min', 'max']].head(10))

## 3. Eksekusi Matriks 24 Pipeline LOPO-CV (Leave-One-Project-Out)

In [ ]:
from lopo_runner import run_all_24_pipelines

# Menjalankan 24 konfigurasi pipeline faktorial pada seluruh 24 lipatan proyek
df_summary = run_all_24_pipelines(output_dir="../results", use_gpu=True)
display(df_summary.head(10))

## 4. Uji Signifikansi Statistik (Friedman & Nemenyi Post-hoc)

In [ ]:
from statistical_tests import run_statistical_analysis

rank_f1 = run_statistical_analysis(raw_results_dir="../results/raw_predictions", metric_col="F1", output_file="../results/friedman_nemenyi_f1_results.csv")
rank_prauc = run_statistical_analysis(raw_results_dir="../results/raw_predictions", metric_col="PR_AUC", output_file="../results/friedman_nemenyi_prauc_results.csv")
display(rank_f1.head(10))

## 5. Visualisasi Publikasi Ilmiah

In [ ]:
from visualizer import generate_all_visualizations

generate_all_visualizations(summary_csv_path="../results/summary_metrics_24_pipelines.csv", plots_dir="../results/plots")
print("Visualisasi selesai dihasilkan di folder results/plots/")